# Figure: Ratios of fugacity coefficients

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure

In [ ]:
# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H","\u03B3<sub>H<sub>2</sub></sub>/\u03B3<sub>H<sub>2</sub>O</sub>"),
    ("C", "\u03B3<sub>CO</sub>/\u03B3<sub>CO<sub>2</sub></sub>"),
    ("S","(\u03B3<sub>S<sub>2</sub></sub>)<sup>0.5</sup>/\u03B3<sub>SO<sub>2</sub></sub>"),
    ('SH',"[\u03B3<sub>SO<sub>2</sub></sub>\u03B3<sub>H<sub>2</sub></sub>]/\u03B3<sub>H<sub>2</sub>S</sub>"),
    ("HC1","\u03B3<sub>CH<sub>4</sub></sub>/[\u03B3<sub>CO</sub>(\u03B3<sub>H<sub>2</sub></sub>)<sup>2</sup>]"),
    ("HC2","\u03B3<sub>CH<sub>4</sub></sub>/[\u03B3<sub>CO<sub>2</sub></sub>(\u03B3<sub>H<sub>2</sub></sub>)<sup>2</sup>]"),
    ('CS',"[\u03B3<sub>OCS</sub>/[\u03B3<sub>CO</sub>(\u03B3<sub>S<sub>2</sub></sub>)<sup>0.5</sup>]")
]

n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.03, horizontal_spacing=0.03,
    subplot_titles=['1100 \u00B0C','1220 \u00B0C','1030 \u00B0C','1200 \u00B0C']
)

for r, (species,y_label) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        lw = 5.5
        for tool in TOOLS:
            lw = lw-0.5
            if tool == 'VESIcal_Iacono':
                continue
            if tool == 'SulfurX':
                if species in ['H','C','S','HC','SC']:
                    continue
            df = systems.get(sample, {}).get(tool)
            if df is None or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            final = len(df)-1
            if df.loc[final,'P_bars'] > 1:
                continue
            if species == 'H':
                y = (df['H2O_v_mf']/(df['H2_v_mf']*((10.**df['logfO2'])**0.5)))/(df.loc[final,'H2O_v_mf']/(df.loc[final,'H2_v_mf']*((10.**df.loc[final,'logfO2'])**0.5)))
            elif species == 'C':
                y = (df['CO2_v_mf']/(df['CO_v_mf']*((10.**df['logfO2'])**0.5)))/(df.loc[final,'CO2_v_mf']/(df.loc[final,'CO_v_mf']*((10.**df.loc[final,'logfO2'])**0.5)))
            elif species == 'S':
                y = ((df['SO2_v_mf']*df['P_bars']**0.5)/(df['S2_v_mf']**0.5*((10.**df['logfO2']))))/(df.loc[final,'SO2_v_mf']/(df.loc[final,'S2_v_mf']**0.5*((10.**df.loc[final,'logfO2']))))
            elif species == 'SH':
                y = ((df['H2S_v_mf']*((10.**df['logfO2'])))/(df['SO2_v_mf']*df['H2_v_mf']*df['P_bars']))/((df.loc[final,'H2S_v_mf']*((10.**df.loc[final,'logfO2'])))/(df.loc[final,'SO2_v_mf']*df.loc[final,'H2_v_mf']))
            elif species == 'HC1':
                y = ((df['CO_v_mf']*df['H2_v_mf']**2.*df['P_bars']**2.)/(df['CH4_v_mf']*(10**df['logfO2'])**0.5))/((df.loc[final,'CO_v_mf']*df.loc[final,'H2_v_mf']**2.)/(df.loc[final,'CH4_v_mf']*(10**df.loc[final,'logfO2'])**0.5))
            elif species == 'HC2':
                y = ((df['CO2_v_mf']*df['H2_v_mf']**2.*df['P_bars']**2.)/(df['CH4_v_mf']*(10**df['logfO2'])))/((df.loc[final,'CO2_v_mf']*df.loc[final,'H2_v_mf']**2.)/(df.loc[final,'CH4_v_mf']*(10**df.loc[final,'logfO2'])))
            elif species in ['CS']:
                if tool in ['Dcompress','EVo']:
                    continue
                y = ((df['CO_v_mf']*df['S2_v_mf']**0.5*df['P_bars']**0.5)/df['OCS_v_mf'])/((df.loc[final,'CO_v_mf']*df.loc[final,'S2_v_mf']**0.5)/df.loc[final,'OCS_v_mf'])
            #x_norm = df["P_bars"] / p_init
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=df['P_bars'], y=y,
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=lw
                              ),
                    showlegend=(r == 1 and c == 2),
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label, row=r, col=1)
        if r == 7:
           fig.update_xaxes(title_text="P (bars)", row=r, col=c, range=[0, None])

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=1, y=0,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=1200, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    fig.write_image("figures/SuppFig_fugacity_coeffs.png", scale=2, height=1200, width=1000)

fig.show()